1. Import pandas API on Spark

In [51]:
# Import pandas API on Spark (Koalas)
# This provides a pandas-like interface, but executions run on Spark.
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"  # MUST be first

import pyspark.pandas as ps


2. Load the bank_customers dataset (CSV version)

In [53]:

from pyspark.sql import SparkSession
import pyspark.pandas as ps
spark = (
    SparkSession.builder
    .appName("pandas-on-spark")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

# Define the source path for the CSV dataset
# This dataset will be loaded using pandas API on Spark
# Define a local demo data directory
base_dir = os.getcwd()  # current notebook folder
csv_path = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "bank_customers"))

# Read the CSV file using ps.read_csv()
# The operation looks like pandas, but it executes through Spark
psdf = ps.read_csv(csv_path,index_col="customer_id")

# Display the first rows to confirm load
psdf.head()



,name,age,account_type,balance,is_active,risk_score
customer_id,,,,,,
1001,Sofia,25,checking,5200.5,True,4
1002,Lukas,34,savings,11200.1,True,6
1003,Amina,29,checking,3400.2,False,3
1004,Omar,41,checking,21800.0,True,5
1005,Kwame,32,savings,670.0,False,6


(Optional) 2B. Load the Parquet version (recommended)

In [54]:
base_dir = os.getcwd()  # current notebook folder
# Use this cell if you're storing data in Parquet format
parquet_path = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "bank_customers_parquet"))

# Parquet is columnar and more efficient with Spark
psdf_parquet = ps.read_parquet(parquet_path)

# Show first rows
psdf_parquet.head()


c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\pandas\utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_parquet`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,customer_id,name,age,account_type,balance,is_active,risk_score
0,1007,Hiroshi,45,checking,7800.0,True,9
1,1010,Sakura,28,checking,21000.9,True,7
2,1001,Sofia,25,checking,5200.5,True,8
3,1003,Amina,29,checking,3400.2,False,6
4,1002,Lukas,34,savings,11200.1,True,2


3. Simple EDA demo (GroupBy + Sort)

(This replicates your slide’s mini-demo but applied to banking data)

In [55]:
# Select only the relevant columns for the analysis
# Group by account_type and aggregate the sum of balances
# Result is a pandas Series backed by Spark execution
summary = (
    psdf[["account_type", "balance"]]
        .groupby("account_type")
        .sum(numeric_only=True)["balance"]
        .sort_values(ascending=False)
)



# Display aggregated results
summary.head()


c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\pandas\groupby.py:893: FutureWarning: Default value of `numeric_only` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(


account_type
savings     65821.1
checking    60701.6
Name: balance, dtype: float64

4. Additional demo: Risk score analysis

In [56]:
# Compute the average risk_score per account_type
# Demonstrates multi-column grouping and metric analysis
risk_stats = (
    psdf.groupby("account_type")
        .risk_score.mean()
        .sort_values(ascending=False)
)

risk_stats


c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\pandas\groupby.py:649: FutureWarning: Default value of `numeric_only` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(


account_type
savings     4.750000
checking    4.333333
Name: risk_score, dtype: float64

5. Converting between APIs
🔄 pandas-on-Spark → Spark DataFrame

In [57]:
# Convert the pandas-on-Spark DataFrame into a native Spark DataFrame
# Useful when you need performance-tuned operations or Spark SQL
spark_df = psdf.to_spark()

# Print schema to show Spark DataFrame structure
spark_df.printSchema()


root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- risk_score: integer (nullable = true)



c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\pandas\utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


🔄 pandas → pandas-on-Spark

In [58]:
# Example Pandas DataFrame
import pandas as pd

pdf = pd.DataFrame({"x": [1, 2, 3], "y": [10, 20, 30]})

# Convert regular Pandas DataFrame into pandas-on-Spark DataFrame
psdf2 = ps.from_pandas(pdf)

psdf2


,x,y
0,1,10
1,2,20
2,3,30


6. Pitfall Example — Row-wise UDF (avoid this)

In [59]:
# BAD PRACTICE:
# Row-wise apply is executed slowly because it breaks vectorized execution
def label_balance(row):
    return "high" if row.balance > 10000 else "low"

# This will trigger a warning and is slow on large datasets
ps.set_option("compute.ops_on_diff_frames", True)

psdf["balance_label_bad"] = psdf.apply(label_balance, axis=1)

psdf.head()


c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\pandas\utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,name,age,account_type,balance,is_active,risk_score,balance_label_bad
customer_id,,,,,,,
1001,Sofia,25,checking,5200.5,True,4,low
1002,Lukas,34,savings,11200.1,True,6,high
1003,Amina,29,checking,3400.2,False,3,low
1004,Omar,41,checking,21800.0,True,5,high
1005,Kwame,32,savings,670.0,False,6,low


7. Recommended vectorized approach

In [60]:
# GOOD PRACTICE:
# Use vectorized column-level operations instead of row-wise apply
# Good practice: vectorized column transformation
psdf["balance_label"] = psdf["balance"].map(
    lambda x: "high" if x > 10000 else "low"
)

psdf[["name", "balance", "balance_label"]].head()



c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:198: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true, but has reached the error below and can not continue. Note that 'spark.sql.execution.arrow.pyspark.fallback.enabled' does not have an effect on failures in the middle of computation.
  An error occurred while calling o4287.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:98)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:94)
	at jdk.internal.reflect.GeneratedMethodAccessor91.invoke(Unknown Source)
	at java.base/jdk.internal.reflect.DelegatingMethodAcces

Py4JJavaError: An error occurred while calling o4287.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:98)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:94)
	at jdk.internal.reflect.GeneratedMethodAccessor91.invoke(Unknown Source)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.exe

c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:198: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true, but has reached the error below and can not continue. Note that 'spark.sql.execution.arrow.pyspark.fallback.enabled' does not have an effect on failures in the middle of computation.
  An error occurred while calling o4323.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:98)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:94)
	at jdk.internal.reflect.GeneratedMethodAccessor91.invoke(Unknown Source)
	at java.base/jdk.internal.reflect.DelegatingMethodAcces

Py4JJavaError: An error occurred while calling o4323.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:98)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:94)
	at jdk.internal.reflect.GeneratedMethodAccessor91.invoke(Unknown Source)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:566)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.exe

8. Even better: Use Spark SQL functions (max performance)

In [69]:
# Convert to Spark DataFrame for optimized transformations
spark_df = psdf.to_spark()

# Use Spark SQL functions for performance and scalability
from pyspark.sql.functions import when, col

spark_df = spark_df.withColumn(
    "balance_label_sql",
    when(col("balance") > 10000, "high").otherwise("low")
)

# Show example rows
spark_df.select("name", "balance", "balance_label_sql").show(5, truncate=False)



+-----+-------+-----------------+
|name |balance|balance_label_sql|
+-----+-------+-----------------+
|Sofia|5200.5 |low              |
|Lukas|11200.1|high             |
|Amina|3400.2 |low              |
|Omar |21800.0|high             |
|Kwame|670.0  |low              |
+-----+-------+-----------------+
only showing top 5 rows



c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\.venv\Lib\site-packages\pyspark\pandas\utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
